In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
csv_file_base = Path("../data/temp/")
csv_filename = "202303-citibike-tripdata_3.csv"

parquet_file_base = Path("../data/raw/")
parquet_filename = csv_filename.split(".")[0] + ".parquet"

columns = ["ride_id", "started_at"]
df = pd.read_csv(csv_file_base / csv_filename, engine="pyarrow", usecols=columns)

# Convert datetime and add date column
df["started_at"] = pd.to_datetime(df["started_at"])
df["ride_date"] = df["started_at"].dt.strftime("%Y-%m-%d")

# Aggregate to get the daily ridership
df_agg = (
    df.groupby("ride_date")
    .agg(total_rides=("ride_id", "nunique"))
    .reset_index()
    .sort_values("ride_date")
)

# Save to parquet
df_agg.to_parquet(parquet_file_base / parquet_filename, compression="gzip", index=False)
print(f"{parquet_filename} processed")

In [ ]:
df_agg